# 03 — MARL Training (Google Colab)

Trains the price agent and the sentiment agent for **one training seed** (`SEED`, default 0)
for `training.total_timesteps` steps each (see `config.yaml`).

Checkpoints are saved every `training.save_frequency` steps to `results/checkpoints/seed{SEED}/`
and copied to `MyDrive/MARL/results/checkpoints/seed{SEED}/` as soon as each one is written.

**Before running**
1. Push the repo (current `config.yaml`, `src/`).
2. On Drive: `MyDrive/MARL/data/processed/{TICKER}_processed.csv` and
   `MyDrive/MARL/data/sentiment/{TICKER}_sentiment.csv` (written by notebook 05) for every ticker.

Train seed 0 first, run the policy check at the end, and stop. Change `SEED` and re-run only after checking.

## Step 1 — Training seed

In [ ]:
SEED = 0

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/MARL"
print("Drive mounted.")

## Step 3 — Clone repo and install dependencies

In [ ]:
import os

if not os.path.exists("/content/MARL"):
    !git clone https://github.com/lww637-cmyk/MARL.git /content/MARL
else:
    !git -C /content/MARL pull
%cd /content/MARL
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

import sys
sys.path.insert(0, "/content/MARL")
print("Dependencies installed.")

## Step 4 — Copy data files from Drive
Fails if any processed or sentiment file is missing, so training never runs without sentiment.

In [ ]:
import shutil
import pandas as pd
from src.training.train_marl import load_config

config  = load_config()
TICKERS = config["data"]["tickers"]
print("Tickers:", TICKERS, "| end_date:", config["data"]["end_date"])

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/sentiment", exist_ok=True)

missing = []
for t in TICKERS:
    for kind in ["processed", "sentiment"]:
        src = f"{DRIVE}/data/{kind}/{t}_{kind}.csv"
        if not os.path.exists(src):
            missing.append(src)
            continue
        shutil.copy(src, f"data/{kind}/{t}_{kind}.csv")
assert not missing, f"Missing on Drive: {missing}"

for t in TICKERS:
    p = pd.read_csv(f"data/processed/{t}_processed.csv", parse_dates=["date"])
    s = pd.read_csv(f"data/sentiment/{t}_sentiment.csv", parse_dates=["date"])
    print(f"{t}: prices {p.date.min().date()}..{p.date.max().date()} ({len(p)} rows) | "
          f"sentiment {s.date.min().date()}..{s.date.max().date()} ({len(s)} days)")

## Step 5 — Validate environment (train split)

In [ ]:
from src.environment.stock_env import StockTradingEnv

env = StockTradingEnv(config, mode="train", split="train")
print(f"Train split: {env.n_steps} trading days "
      f"({env.common_dates[0].date()} .. {env.common_dates[-1].date()})")
print("Price obs:", env.observation_space("price_agent").shape,
      "| Sentiment obs:", env.observation_space("sentiment_agent").shape)

for t in TICKERS:
    d = env.dataset[t].loc[env.common_dates]
    print(f"  {t}: has_sentiment on {d.has_sentiment.mean()*100:.1f}% of train days")
    assert d.has_sentiment.mean() > 0, f"{t} has no sentiment in the train split"

## Step 6 — Train one seed (resumes automatically)
Every checkpoint is copied to Drive as soon as it is saved. If Colab disconnects, re-run the notebook from the top: each agent continues from its newest checkpoint in `MyDrive/MARL/results/checkpoints/seed{SEED}/`, and an agent with a `_final.zip` there is skipped.

In [ ]:
from src.training.train_marl import find_checkpoints

DRIVE_CKPT = f"{DRIVE}/results/checkpoints/seed{SEED}"
TOTAL      = config["training"]["total_timesteps"]

for agent_id in ["price_agent", "sentiment_agent"]:
    found = find_checkpoints(agent_id, [DRIVE_CKPT], TOTAL)
    if found:
        step, path = found[0]
        print(f"{agent_id}: will resume from step {step:,} of {TOTAL:,} ({os.path.basename(path)})")
    else:
        print(f"{agent_id}: no checkpoint on Drive — starts from step 0")

In [ ]:
from src.training.train_marl import train

price_agent, sentiment_agent = train(config, seed=SEED, copy_dir=DRIVE_CKPT)
print("Training complete. Checkpoints on Drive:")
print(sorted(os.listdir(DRIVE_CKPT)))

## Step 7 — Policy check (train split only), then STOP
Deterministic actions of each final model on every train-split day. A per-asset std near 0 means the policy ignores its inputs (e.g. a constant uniform allocation).

In [ ]:
import numpy as np
from stable_baselines3 import PPO

env = StockTradingEnv(config, mode="eval", split="train")
env.reset(seed=0)
cols = TICKERS + ["cash"]

for agent_id in ["price_agent", "sentiment_agent"]:
    model = PPO.load(f"results/checkpoints/seed{SEED}/{agent_id}_final.zip")
    weights = []
    for i in range(env.n_steps):
        env.start_idx, env.timestep = i, 0
        a, _ = model.predict(env.observe(agent_id), deterministic=True)
        a = np.clip(a, 0.0, 1.0)
        weights.append(a / (a.sum() + 1e-8))
    w = pd.DataFrame(weights, columns=cols)
    print(f"\n{agent_id} — allocation over {len(w)} train days")
    print(pd.DataFrame({"mean": w.mean(), "std": w.std()}).round(3).T.to_string())

**Stop here.** Check the policy above before training more seeds. For the next seed, set `SEED` in Step 1 and re-run from Step 1.